# RiboUnmix: model checks and prediction inspection

A CPU walkthrough of the **actual RiboUnmix implementation**. No downloaded dataset, trained checkpoint, or GPU is required for the core cells.

1. Build a small model and two toy transcripts observed in two datasets.
2. Check shapes, masking, positivity, mean normalization, target scaling, and reference invariance.
3. Check NB2 values against an independent distribution implementation and backpropagate.
4. Optionally inspect saved predictions and loss-ablation summaries.

The model below has random weights. Its curves are **not trained results**, and a successful sanity check says nothing about biological recovery or generalization. Full training stays in the Hydra entrypoints.

Install from the repository root: `python -m pip install -r requirements-notebooks.txt`. Select that environment as the notebook kernel, then **Restart Kernel → Run All**.


In [ ]:
# Paths are resolved from the repository root or any of its subdirectories.
from pathlib import Path
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "config").is_dir() and (p / "Models").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the RiboUnmix checkout.")
if not (ROOT / "Models/RiboUnmixModel/RiboUnmixModel.py").is_file():
    raise FileNotFoundError(
        "This checkout lacks the canonical model. See Docs/repository_alignment.md."
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import copy
import importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import yaml
from IPython.display import display
from torch.nn.utils.rnn import pack_padded_sequence
from Models.RiboUnmixModel import RiboUnmixModel
from Models.utils.stable_numerics import masked_mean, nb2_nll_from_log_mean

display(pd.Series({
    name: importlib.metadata.version(name)
    for name in ("torch", "numpy", "pandas", "matplotlib", "PyYAML")
}, name="Installed version"))


In [ ]:
SEED = 42
# Leave these as None for a self-contained CPU run.
PREDICTION_PATH = None  # Path("results/.../predictions_test_best_nb_nll_....parquet")
ABLATION_TABLE_DIR = None  # Path("analyses/artifacts/benchmarking/loss_ablation_v1/best_nb_nll")
MAX_PREDICTION_ROWS = 64
PROFILE_INDEX = 0  # Fixed example index; not selected by best performance.
# Match these to the selected checkpoint's frozen loss configuration.
NB_LOG_ALPHA_BOUNDS = (-5.0, 1.0)

torch.manual_seed(SEED)
torch.set_num_threads(2)
rng = np.random.default_rng(SEED)
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.18,
})


## 1. A small version of the real model

The runnable real-data configuration is the starting point; the cell explicitly reduces the hidden sizes, uses two dataset IDs, disables optional input features, and sets dropout to zero. This is an educational configuration, not a reproduction of the paper.

The shared profile is sequence-only and mean-one. The observation mean uses the **observed target mean** as its scale:
\[
S_{dt}=\operatorname{mean}_{i\in V_t}y_{dti},\qquad
\mu_{dti}=S_{dt}\frac{\gamma_{dti}L_{ti}}{\operatorname{mean}_{j\in V_t}(\gamma_{dtj}L_{tj})}.
\]
The denominator is present only when `mass_conservation=True`. The benchmark and synthetic defaults disable it. Neither variant predicts absolute abundance from sequence alone.


In [ ]:
config = yaml.safe_load((ROOT / "config/config_ribounmix_multidataset.yaml").read_text())
model_config = copy.deepcopy(config["model"])
model_config["mass_conservation"] = True
model_config["additional_sequence_features"] = {}
model_config["biological_params"].update(hidden_size=16, num_layers=1, dropout=0.0)
model_config["gamma_centering"] = {
    "mode": "fixed_reference",
    "dataset_constant_scale_gauge": "geometric_mean_one",
    "reference": {"weighting": "equal", "minimum_datasets": 2, "chunk_size": 2},
}
bias_config = model_config["dataset_bias_params"]
bias_config.update(
    num_datasets=2, dataset_embeddings_size=4, codon_embeddings_size=4,
    context_gru_hidden_size=8, context_gru_num_layers=1, context_gru_dropout=0.0,
    context_gru_tbptt_window=0,
)
for key in ("dataset_multiplicative_allocation_bias_submodule_params",
            "dataset_log_sigma_submodule_params"):
    bias_config[key].update(hidden_size=8, dropout=0.0)

encoding_dir = ROOT / "Datasets/encodings"
encodings = {
    key: yaml.safe_load((encoding_dir / filename).read_text())
    for key, filename in {
        "nt_encoding": "nt_encoding.yaml", "codon_to_aa_encoding": "codon2aa.yaml",
        "codon_encoding": "codon_encoding.yaml", "aa_encoding": "aa_encoding.yaml",
    }.items()
}
model = RiboUnmixModel(
    model_config,
    selected_dataset_names=["toy_A", "toy_B"], selected_dataset_ids=[0, 1],
    reference_dataset_names=["toy_A", "toy_B"], reference_dataset_ids=[0, 1],
    reference_dataset_quality_weights=[1.0, 1.0],
    **encodings,
).cpu().eval()
# Neutral gamma initialization would make the gauge tests trivial.
# Perturb only this disposable toy model to exercise a nonconstant correction.
with torch.no_grad():
    torch.nn.init.normal_(
        model.dataset_bias_model.observation_bias_head.log_bias_head.weight, std=0.1
    )
print(f"{sum(p.numel() for p in model.parameters()):,} parameters; CPU; random weights")


## 2. Construct aligned inputs

Rows are transcript–dataset pairs, with two lengths to exercise padding. Shapes are:
- sequence features: `[B, T, 97]` = 12 nucleotide + 64 codon + 21 amino-acid channels;
- codon IDs, mask, target: `[B, T]`;
- dataset IDs: `[B]`.

Toy counts use NB2 noise with $\operatorname{Var}(Y)=\mu+\alpha\mu^2$. The two observations of a transcript share their sequence. Real loader code enforces that same identity.


In [ ]:
codon_map = encodings["codon_encoding"]
aa_map = encodings["aa_encoding"]
feature_lut = torch.zeros(len(codon_map), 97)
for codon, index in codon_map.items():
    nt = torch.tensor([v for base in codon for v in encodings["nt_encoding"][base]])
    codon_onehot = torch.nn.functional.one_hot(torch.tensor(index), len(codon_map))
    aa_id = aa_map[encodings["codon_to_aa_encoding"][codon]]
    aa_onehot = torch.nn.functional.one_hot(torch.tensor(aa_id), len(aa_map))
    feature_lut[index] = torch.cat([nt, codon_onehot, aa_onehot]).float()

lengths = torch.tensor([48, 48, 32, 32])
sample_ids = ["transcript_A", "transcript_A", "transcript_B", "transcript_B"]
dataset_ids = torch.tensor([0, 1, 0, 1])
mask = torch.arange(48)[None, :] < lengths[:, None]
codon_ids = torch.zeros(4, 48, dtype=torch.long)
target = torch.zeros(4, 48)
sense_ids = [v for c, v in codon_map.items() if encodings["codon_to_aa_encoding"][c] != "*"]
for start, length in [(0, 48), (2, 32)]:
    codons = torch.tensor(rng.choice(sense_ids, size=length))
    codon_ids[start:start + 2, :length] = codons
    position = np.linspace(0, 1, length)
    shared = 1.0 + 1.5 * np.exp(-((position - 0.55) / 0.1) ** 2)
    for dataset in (0, 1):
        correction = np.exp((1 if dataset == 0 else -1) * 0.5 * np.cos(2 * np.pi * position))
        shape = shared * correction
        true_mu = (8 + 4 * dataset) * shape / shape.mean()
        alpha = 0.15
        counts = rng.negative_binomial(1 / alpha, 1 / (1 + alpha * true_mu))
        target[start + dataset, :length] = torch.tensor(counts, dtype=torch.float32)
features = feature_lut[codon_ids] * mask.unsqueeze(-1)

def forward_rows(rows: list[int], values: torch.Tensor):
    """Evaluate selected pair rows; values is a full [B,T] target tensor."""
    row_ids = torch.tensor(rows, dtype=torch.long)
    packed = pack_padded_sequence(
        features[row_ids], lengths[row_ids], batch_first=True, enforce_sorted=False
    )
    return model(
        x_packed=packed, codon_ids=codon_ids[row_ids], id_datasets=dataset_ids[row_ids],
        mask=mask[row_ids], target=values[row_ids],
        sample_ids=[sample_ids[i] for i in rows],
    )

with torch.no_grad():
    mu, log_alpha, extra = forward_rows(list(range(4)), target)

display(pd.DataFrame({
    "transcript": sample_ids, "dataset": dataset_ids.numpy(),
    "valid_codons": lengths.numpy(),
    "target_mean": masked_mean(target, mask).squeeze(1).numpy(),
    "predicted_mean": masked_mean(mu, mask).squeeze(1).numpy(),
}))


## 3. Test properties that should hold

These checks verify finite values, positive valid-position means, normalization, shared-profile agreement, and both gamma gauges. Padding may contain neutral placeholders; metrics must always use the mask.

The centering constraints define a reference-dependent factorization. They do **not** establish that the shared component is purely biological or that the dataset component is purely technical.


In [ ]:
assert mu.shape == log_alpha.shape == mask.shape
for value in (mu[mask], log_alpha[mask], extra["L_bio"][mask], extra["gamma"][mask]):
    assert torch.isfinite(value).all()
assert (mu[mask] > 0).all() and (extra["gamma"][mask] > 0).all()
assert extra["log_gamma"][mask].abs().max() > 1e-6  # Nontrivial test.
torch.testing.assert_close(masked_mean(extra["L_bio"], mask), torch.ones(4, 1))
torch.testing.assert_close(masked_mean(mu, mask), masked_mean(target, mask))
for pair in ([0, 1], [2, 3]):
    torch.testing.assert_close(extra["L_bio"][pair[0]], extra["L_bio"][pair[1]])
    valid = mask[pair[0]]
    torch.testing.assert_close(
        extra["log_gamma"][pair][:, valid].mean(dim=0),
        torch.zeros(int(valid.sum())), atol=2e-6, rtol=0,
    )
torch.testing.assert_close(
    masked_mean(extra["log_gamma"], mask), torch.zeros(4, 1), atol=2e-6, rtol=0
)
print("PASS: shapes, finite outputs, positivity, shared profile, mass and gamma gauges.")


In [ ]:
# Changing only padded targets must not affect valid predictions.
padded_target = torch.where(mask, target, torch.full_like(target, 1e6))
with torch.no_grad():
    padded_mu, _, _ = forward_rows(list(range(4)), padded_target)
    scaled_mu, _, scaled_extra = forward_rows(list(range(4)), 3 * target)
    single_mu, _, single_extra = forward_rows([0], target)
torch.testing.assert_close(padded_mu[mask], mu[mask])
torch.testing.assert_close(scaled_mu[mask], 3 * mu[mask])
torch.testing.assert_close(scaled_extra["L_bio"], extra["L_bio"])
torch.testing.assert_close(scaled_extra["gamma"], extra["gamma"])
torch.testing.assert_close(single_mu[0, mask[0]], mu[0, mask[0]], atol=2e-5, rtol=2e-5)
torch.testing.assert_close(
    single_extra["log_gamma"][0, mask[0]], extra["log_gamma"][0, mask[0]],
    atol=2e-6, rtol=2e-5,
)
print("PASS: masked targets, target scaling, and singleton fixed-reference inference.")


The scaling check is scientifically important: multiplying the supplied target by three multiplies the predicted mean by three while leaving the learned shape unchanged. Absolute-count evaluation therefore uses observed target scale. Sequence-only interpretation should focus on `L_bio`, and comparisons with other methods must account for access to that scale.


In [ ]:
# The non-mass-conserving branch has a different, still testable identity.
model.mass_conservation = False
try:
    with torch.no_grad():
        free_mu, _, free_extra = forward_rows(list(range(4)), target)
    expected = masked_mean(target, mask) * free_extra["gamma"] * free_extra["L_bio"]
    torch.testing.assert_close(free_mu[mask], expected[mask])
finally:
    model.mass_conservation = True
print("PASS: with mass conservation disabled, mu = S × gamma × L_bio on valid positions.")


## 4. Check likelihood values and gradients

This cell checks the **NB2 component**, not the full replica-weighted, correlation-regularized training objective. It uses the project's stable implementation and compares it with PyTorch's negative-binomial distribution on integer counts. `log_sigma` in saved files is the historical name of **log dispersion alpha**, not log standard deviation.


In [ ]:
model.zero_grad(set_to_none=True)
mu_grad, log_alpha_grad, grad_extra = forward_rows(list(range(4)), target)
bounded_log_alpha = log_alpha_grad.clamp(*NB_LOG_ALPHA_BOUNDS)
element_nll = nb2_nll_from_log_mean(target, grad_extra["log_mu"], bounded_log_alpha)
loss = element_nll[mask].mean()

# NB2: size r=1/alpha, torch NB logits=log(alpha*mu), giving E[Y]=mu.
reference = torch.distributions.NegativeBinomial(
    total_count=(-bounded_log_alpha[mask]).exp(),
    logits=grad_extra["log_mu"][mask] + bounded_log_alpha[mask],
)
torch.testing.assert_close(
    element_nll[mask], -reference.log_prob(target[mask]), atol=2e-5, rtol=2e-5
)
loss.backward()
gradients = [p.grad for p in model.parameters() if p.grad is not None]
assert gradients and all(torch.isfinite(g).all() for g in gradients)
gradient_norm = torch.sqrt(sum(g.square().sum() for g in gradients))
assert gradient_norm > 0
display(pd.Series({
    "NB2 mean positional NLL": float(loss.detach()),
    "gradient L2 norm": float(gradient_norm),
    "parameter tensors receiving gradients": len(gradients),
}, name="One forward/backward check — no optimizer update"))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), constrained_layout=True)
position = np.arange(48)
axes[0].plot(position, target[0].numpy(), color="#64748b", alpha=0.65, label="Toy observed counts")
axes[0].plot(position, mu[0].numpy(), color="#0f766e", label="Random model mean")
axes[0].set(title="Observation space", xlabel="Codon position", ylabel="Counts")
axes[0].legend(fontsize=8)
axes[1].plot(position, extra["L_bio"][0].numpy(), color="#0f766e")
axes[1].axhline(1, color="#94a3b8", linestyle="--")
axes[1].set(title="Shared profile (random weights)", xlabel="Codon position", ylabel="Mean-one load")
for row, color in [(0, "#2563eb"), (1, "#c2410c")]:
    axes[2].plot(position, extra["gamma"][row].numpy(), color=color, label=f"Toy dataset {row}")
axes[2].set(title="Dataset corrections", xlabel="Codon position", ylabel="Gamma")
axes[2].legend(fontsize=8)
fig.suptitle("Sanity-check outputs, not trained recovery results", fontsize=12)
plt.show()


## 5. Optional: inspect a real prediction export

Set `PREDICTION_PATH` in the parameters cell. Only the first `MAX_PREDICTION_ROWS` rows are loaded for an interactive preview; this is **not a representative benchmark**.

Metrics use raw PCC, **log1p PCC with the same transform for every model**, and mean positional NB2 NLL. Undefined correlations remain missing, with valid counts reported. The NLL here is unweighted; it is not the length-weighted endpoint in the loss-ablation report. Match the dispersion bounds to the frozen run configuration before interpreting it.

For definitive comparisons, use all held-out transcripts, identical IDs/targets/masks, the same checkpoint-selection rule, and separate seed-level uncertainty from transcript-level uncertainty.


In [ ]:
def pearson_valid(x: np.ndarray, y: np.ndarray) -> float:
    """Pearson correlation; undefined for fewer than two or constant positions."""
    x, y = x - x.mean(), y - y.mean()
    denominator = np.linalg.norm(x) * np.linalg.norm(y)
    if x.size < 2 or denominator <= 1e-12:
        return float("nan")
    return float(np.dot(x, y) / denominator)

def preview_metrics(row: dict) -> dict:
    """Compute metrics for one exported [T] profile, preserving its valid mask."""
    y, prediction = (np.asarray(row[k], dtype=np.float64) for k in ("target", "mu"))
    valid = np.asarray(row["mask"], dtype=bool)
    log_dispersion = np.broadcast_to(np.asarray(row["log_sigma"], dtype=np.float64), y.shape)
    if y.ndim != 1 or prediction.shape != y.shape or valid.shape != y.shape or not valid.any():
        raise ValueError(f"Malformed profile or empty mask: {row['transcript_id']}")
    y, prediction, log_dispersion = y[valid], prediction[valid], log_dispersion[valid]
    if not all(np.isfinite(v).all() for v in (y, prediction, log_dispersion)):
        raise ValueError("Non-finite valid values in prediction export.")
    if (y < 0).any() or (prediction <= 0).any():
        raise ValueError("Expected nonnegative targets and strictly positive predictions.")
    nll = nb2_nll_from_log_mean(
        torch.from_numpy(y), torch.from_numpy(np.log(prediction)),
        torch.from_numpy(np.clip(log_dispersion, *NB_LOG_ALPHA_BOUNDS)),
    ).mean().item()
    return {
        "transcript_id": str(row["transcript_id"]), "valid_codons": len(y),
        "raw_pcc": pearson_valid(prediction, y),
        "log1p_pcc": pearson_valid(np.log1p(prediction), np.log1p(y)),
        "nb2_nll_position_mean": nll,
        "mean_ratio": prediction.mean() / y.mean() if y.mean() > 0 else np.nan,
    }

prediction_rows = []
if PREDICTION_PATH is None:
    print("Optional prediction inspection skipped. Set PREDICTION_PATH to enable.")
else:
    import pyarrow.parquet as pq
    path = Path(PREDICTION_PATH)
    path = path if path.is_absolute() else ROOT / path
    if MAX_PREDICTION_ROWS < 1:
        raise ValueError("MAX_PREDICTION_ROWS must be positive.")
    parquet = pq.ParquetFile(path)
    batch = next(parquet.iter_batches(
        batch_size=MAX_PREDICTION_ROWS,
        columns=["transcript_id", "target", "mu", "mask", "log_sigma"],
    ), None)
    if batch is None:
        raise ValueError("The prediction export is empty.")
    prediction_rows = batch.to_pylist()
    preview = pd.DataFrame([preview_metrics(row) for row in prediction_rows])
    print(f"Preview: {len(preview)} of {parquet.metadata.num_rows} rows — {path.name}")
    display(preview.head(10))
    display(preview.select_dtypes("number").agg(["count", "mean", "median"]))


In [ ]:
if prediction_rows:
    row = prediction_rows[PROFILE_INDEX]
    valid = np.asarray(row["mask"], dtype=bool)
    positions = np.flatnonzero(valid)
    y, prediction = [np.asarray(row[k], dtype=float)[valid] for k in ("target", "mu")]
    fig, ax = plt.subplots(figsize=(12, 3.5), constrained_layout=True)
    ax.plot(positions, y, color="#64748b", alpha=0.65, label="Observed")
    ax.plot(positions, prediction, color="#0f766e", label="Predicted mean")
    ax.set(title=f"Saved prediction: {row['transcript_id']}",
           xlabel="Codon position", ylabel="Counts")
    ax.legend()
    plt.show()


## 6. Optional: compare loss-ablation seeds

Set `ABLATION_TABLE_DIR` to the directory regenerated by
`analyses/analyze_benchmark_loss_ablation.py --training-seeds 42,43,44 --require-complete`.

The tables below show raw PCC and NB2 NLL with a positive-is-better convention for the full objective. They display **individual fitted seeds** and their range. The range is descriptive, not a confidence interval. Model-specific NB-VST PCC is omitted here because each model transforms the target with its own dispersion.


In [ ]:
if ABLATION_TABLE_DIR is None:
    print("Optional ablation comparison skipped. Set ABLATION_TABLE_DIR to enable.")
else:
    table_dir = Path(ABLATION_TABLE_DIR)
    table_dir = table_dir if table_dir.is_absolute() else ROOT / table_dir
    effects = pd.read_csv(table_dir / "seed_specific_effects.csv")
    effects = effects[effects["metric"].isin(["raw_pcc", "nb2_nll"])].copy()
    effects["full_advantage"] = np.where(
        effects["metric"].eq("nb2_nll"), effects["estimate"], -effects["estimate"]
    )
    # Never silently collapse multiple entries for the same comparison.
    if effects.duplicated(["dataset", "arm", "metric", "training_seed"]).any():
        raise ValueError("Duplicate seed-comparison rows in the effects table.")
    display(effects.pivot(
        index=["dataset", "arm", "metric"], columns="training_seed", values="full_advantage"
    ))
    display(effects.groupby(["dataset", "arm", "metric"])["full_advantage"].agg(
        seed_count="count", mean="mean", minimum="min", maximum="max"
    ))
    nb_only = effects[effects["arm"].eq("nb_only")]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    datasets = sorted(nb_only["dataset"].unique())
    for ax, metric in zip(axes, ["raw_pcc", "nb2_nll"]):
        for index, dataset in enumerate(datasets):
            values = nb_only.loc[
                nb_only["dataset"].eq(dataset) & nb_only["metric"].eq(metric),
                "full_advantage",
            ].to_numpy()
            ax.scatter(np.full(len(values), index), values, color="#0f766e", alpha=0.8)
        ax.axhline(0, color="#64748b", linestyle="--")
        ax.set(xticks=range(len(datasets)), xticklabels=[d.split("_")[0] for d in datasets],
               title=metric, ylabel="Full advantage over NB-only")
    plt.show()


## What to do next

- **Train or evaluate a checkpoint:** use the matching Hydra entrypoint and the checkpoint's frozen model, dataset-ID map, split, and reference-panel configuration. The toy model above is not checkpoint-compatible with a full-size training run.
- **Evaluate generalization:** keep checkpoint selection on validation data and report untouched test data. Default multidataset validation predictions are not an independent test set.
- **Test a scientific claim:** compare matched arms across seeds and splits. Passing the implementation checks above is not evidence for causal biological/technical separation.
- **Run the focused regression checks:** from the repository root, use `python -m unittest discover -s Tests -p 'test_gamma_centering.py'` and `python -m unittest discover -s Tests -p 'test_stable_training_numerics.py'`.
- **Reproduce figures and reports:** see [the analysis index](../analyses/README.md).
